In [1]:
!pip install tf-keras-vis tensorflow==2.20 matplotlib numpy pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.5/52.5 kB 2.4 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
os.makedirs('app', exist_ok=True)
os.makedirs('utils', exist_ok=True)
os.makedirs('assets', exist_ok=True)
print("Folders created ✅")

Folders created ✅


In [4]:
%%writefile utils/gradcam.py

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

CLASS_NAMES = ['Earthquake', 'Fire', 'Flood', 'Normal']

def load_and_preprocess(image_path, target_size=(224, 224)):
    img = tf.keras.utils.load_img(image_path, target_size=target_size)
    img_array = tf.keras.utils.img_to_array(img) / 255.0
    return img_array, np.expand_dims(img_array, axis=0)

def predict_and_explain(model, image_path):
    img_array, img_tensor = load_and_preprocess(image_path)
    preds = model.predict(img_tensor, verbose=0)[0]
    class_idx = np.argmax(preds)
    heatmap = np.random.rand(224, 224)
    overlay = img_array
    return {
        'class': CLASS_NAMES[class_idx],
        'confidence': float(preds[class_idx]),
        'all_probs': {CLASS_NAMES[i]: float(preds[i]) for i in range(4)},
        'heatmap': heatmap,
        'overlay': overlay,
        'original': img_array
    }

Writing utils/gradcam.py


In [5]:
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0

base = EfficientNetB0(include_top=False, weights='imagenet', input_shape=(224,224,3))
x = layers.GlobalAveragePooling2D()(base.output)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.3)(x)
output = layers.Dense(4, activation='softmax')(x)
placeholder_model = models.Model(base.input, output)

print("Placeholder model ready ✅")

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Placeholder model ready ✅


In [6]:
%%writefile app/app.py

import streamlit as st
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from PIL import Image
import tempfile
import os
import sys

sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from utils.gradcam import predict_and_explain

# ── Page config ──────────────────────────────────────────────
st.set_page_config(
    page_title="Disaster Severity Assessment",
    page_icon="🛰️",
    layout="wide"
)

CLASS_NAMES = ['Earthquake', 'Fire', 'Flood', 'Normal']
CLASS_COLORS = {
    'Earthquake': '#e74c3c',
    'Fire':       '#e67e22',
    'Flood':      '#3498db',
    'Normal':     '#2ecc71'
}
CLASS_EMOJI = {
    'Earthquake': '🏚️',
    'Fire':       '🔥',
    'Flood':      '🌊',
    'Normal':     '✅'
}

@st.cache_resource
def load_model():
    model_path = os.path.join(
        os.path.dirname(__file__), '..', 'models', 'efficientnet_phase1.h5'
    )
    return tf.keras.models.load_model(model_path)

st.sidebar.title("🛰️ Disaster Severity Assessment")
st.sidebar.markdown("Upload an aerial image to classify the disaster type and visualize model attention.")
st.sidebar.markdown("---")
st.sidebar.markdown("**Classes:**")
for cls, emoji in CLASS_EMOJI.items():
    st.sidebar.markdown(f"{emoji} {cls}")

st.title("🛰️ Disaster Severity Assessment")
st.markdown("Powered by **EfficientNetB0** + **Grad-CAM** | AIDERv2 Dataset")
st.markdown("---")

uploaded_file = st.file_uploader(
    "Upload an aerial image (PNG/JPG)",
    type=["png", "jpg", "jpeg"]
)

if uploaded_file is not None:
    with tempfile.NamedTemporaryFile(delete=False, suffix='.png') as tmp:
        tmp.write(uploaded_file.read())
        tmp_path = tmp.name

    try:
        model = load_model()
    except Exception:
        st.warning("⚠️ Model not found — using placeholder for UI demo.")
        from tensorflow.keras import layers, models
        from tensorflow.keras.applications import EfficientNetB0
        base = EfficientNetB0(include_top=False, weights='imagenet', input_shape=(224,224,3))
        x = layers.GlobalAveragePooling2D()(base.output)
        x = layers.Dense(256, activation='relu')(x)
        x = layers.Dropout(0.3)(x)
        out = layers.Dense(4, activation='softmax')(x)
        model = models.Model(base.input, out)

    with st.spinner("Analyzing image..."):
        result = predict_and_explain(model, tmp_path)

    col1, col2, col3 = st.columns(3)

    with col1:
        st.subheader("📷 Original Image")
        st.image(result['original'], use_column_width=True)

    with col2:
        st.subheader("🔥 Grad-CAM Heatmap")
        fig, ax = plt.subplots()
        ax.imshow(result['heatmap'], cmap='jet')
        ax.axis('off')
        st.pyplot(fig)

    with col3:
        st.subheader("🔀 Overlay")
        st.image(result['overlay'], use_column_width=True)

    st.markdown("---")

    pred_class = result['class']
    confidence = result['confidence']
    color = CLASS_COLORS[pred_class]
    emoji = CLASS_EMOJI[pred_class]

    st.markdown(f"""
    <div style='background-color:{color}22; border-left: 5px solid {color};
    padding: 16px; border-radius: 8px;'>
        <h2 style='color:{color}'>{emoji} Predicted: {pred_class}</h2>
        <h3>Confidence: {confidence:.2%}</h3>
    </div>
    """, unsafe_allow_html=True)

    st.markdown("---")

    st.subheader("📊 Class Probabilities")
    probs = result['all_probs']
    fig = go.Figure(go.Bar(
        x=list(probs.keys()),
        y=list(probs.values()),
        marker_color=[CLASS_COLORS[c] for c in probs.keys()],
        text=[f"{v:.2%}" for v in probs.values()],
        textposition='outside'
    ))
    fig.update_layout(
        yaxis=dict(range=[0, 1], tickformat='.0%'),
        height=350,
        margin=dict(t=20, b=20)
    )
    st.plotly_chart(fig, use_container_width=True)

    os.unlink(tmp_path)

else:
    st.info("👆 Upload an aerial disaster image to get started.")

Writing app/app.py


In [7]:
!pip install pyngrok

In [8]:
!pip install streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 72.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 118.0 MB/s eta 0:00:00


In [9]:
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴
added 22 packages in 3s
⠴
⠴3 packages are looking for funding
⠴  run `npm fund` for details
⠴

In [10]:
import subprocess
import time

proc = subprocess.Popen(
    ['streamlit', 'run', 'app/app.py', '--server.port=8501', '--server.headless=true'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(5)

tunnel = subprocess.Popen(
    ['lt', '--port', '8501'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(3)

output = tunnel.stdout.readline().decode('utf-8').strip()
print("🚀 Dashboard is live at:", output)

🚀 Dashboard is live at: your url is: https://late-mails-camp.loca.lt


In [11]:
import urllib.request
print(urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())

34.125.73.11


In [12]:
import subprocess
import time

# Kill previous streamlit
subprocess.run(['pkill', '-f', 'streamlit'], capture_output=True)
time.sleep(2)

proc = subprocess.Popen(
    ['streamlit', 'run', 'app/app.py',
     '--server.port=8501',
     '--server.headless=true',
     '--server.enableCORS=false',
     '--server.enableXsrfProtection=false'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(5)
print("✅ Streamlit is running!")
print("👉 Click the link in the top-right of Colab: 'Connect' button > 'View port 8501'")

✅ Streamlit is running!
👉 Click the link in the top-right of Colab: 'Connect' button > 'View port 8501'


In [13]:
from google.colab.output import eval_js
print(eval_js("google.colab.kernel.proxyPort(8501)"))

https://8501-gpu-t4-s-kkb-usw4b0-1ijsny2rq0z9l-b.us-west4-0.prod.colab.dev
